Some images have a blue hue. How many are there and should we convert them to greyscale?

In [ ]:
import os
from PIL import Image
import numpy as np
import pandas as pd
import tensorflow as tf

MAIN_DIR = "/home/onuro/code/simonwilliams32/MRI_project"



FINAL_DATASET_DIR = os.path.join(MAIN_DIR, "raw_data", "final_brain_tumor_preprocessed_dataset")

TRAIN_DIR = FINAL_DATASET_DIR + "/train"
VAL_DIR = FINAL_DATASET_DIR + "/val"
TEST_DIR = FINAL_DATASET_DIR + "/test"
EXTERNAL_VAL_DIR = FINAL_DATASET_DIR + "/external_val"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

class_names = ["glioma", "meningioma", "notumor", "pituitary"]
IMAGE_EXTS = (".png", ".jpg", ".jpeg")

##### Convert blueish images into gray

def is_grayscale(path):
    """Returns True if image is black & white (all RGB channels equal)."""
    try:
        img = Image.open(path).convert("RGB")
        arr = np.array(img)
        return np.array_equal(arr[:, :, 0], arr[:, :, 1]) and np.array_equal(arr[:, :, 1], arr[:, :, 2])
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return None

results = []

for split in os.listdir(FINAL_DATASET_DIR):
    split_path = os.path.join(FINAL_DATASET_DIR, split)
    if not os.path.isdir(split_path):
        continue

    for class_name in os.listdir(split_path):
        class_path = os.path.join(split_path, class_name)
        if not os.path.isdir(class_path):
            continue

        for fname in os.listdir(class_path):
            if not fname.lower().endswith(IMAGE_EXTS):
                continue
            fpath = os.path.join(class_path, fname)
            gray = is_grayscale(fpath)
            results.append({
                "split": split,
                "class": class_name,
                "filename": fname,
                "path": fpath,
                "is_grayscale": gray
            })

df = pd.DataFrame(results)

print(f"Total images checked: {len(df)}")
print(f"Grayscale: {(df['is_grayscale'] == True).sum()}")
print(f"Colormapped/Color: {(df['is_grayscale'] == False).sum()}")
print(f"Errors/unreadable: {df['is_grayscale'].isna().sum()}")

print("\nBreakdown by split and class:")
summary = df.groupby(["split", "class"])["is_grayscale"].value_counts().unstack(fill_value=0)
print(summary)

def convert_to_grayscale_and_save(path):
    """Convert an image to grayscale (L), then save it back as 3-channel RGB
    so shape/format stays consistent with the rest of the pipeline."""
    try:
        img = Image.open(path).convert("RGB")
        gray = img.convert("L")
        gray_rgb = Image.merge("RGB", (gray, gray, gray))
        gray_rgb.save(path)
        return True
    except Exception as e:
        print(f"Error converting {path}: {e}")
        return False

# Only touch the ones flagged as colorized
to_fix = df[df["is_grayscale"] == False]
print(f"\nConverting {len(to_fix)} colorized images to grayscale...")

converted = 0
failed = 0
for path in to_fix["path"]:
    if convert_to_grayscale_and_save(path):
        converted += 1
    else:
        failed += 1

print(f"Converted: {converted}")
print(f"Failed: {failed}")


# load datasets

train_ds = tf.keras.utils.image_dataset_from_directory(
     TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
     label_mode="categorical", shuffle=True, class_names=class_names
 )
val_ds = tf.keras.utils.image_dataset_from_directory(
     VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
     label_mode="categorical", shuffle=False, class_names=class_names
 )
test_ds = tf.keras.utils.image_dataset_from_directory(
     TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
     label_mode="categorical", shuffle=False, class_names=class_names
)
external_val_ds = tf.keras.utils.image_dataset_from_directory(
     EXTERNAL_VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
     label_mode="categorical", shuffle=False, class_names=class_names
 )


print("Training paths prepared:")
print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR:", VAL_DIR)
print("TEST_DIR:", TEST_DIR)
print("EXTERNAL_VAL_DIR:", EXTERNAL_VAL_DIR)

Total images checked: 14602
Grayscale: 14602
Colormapped/Color: 0
Errors/unreadable: 0

Breakdown by split and class:
is_grayscale             True
split        class           
external_val glioma      2004
             meningioma  2004
test         glioma       561
             meningioma   365
             notumor      260
             pituitary    404
train        glioma      2616
             meningioma  1701
             notumor     1212
             pituitary   1886
val          glioma       561
             meningioma   364
             notumor      260
             pituitary    404

Converting 0 colorized images to grayscale...
Converted: 0
Failed: 0
Found 7415 files belonging to 4 classes.
Found 1589 files belonging to 4 classes.
Found 1590 files belonging to 4 classes.
Found 4008 files belonging to 4 classes.
Training paths prepared:
TRAIN_DIR: /home/onuro/code/simonwilliams32/MRI_project/raw_data/final_brain_tumor_preprocessed_dataset/train
VAL_DIR: /home/onuro/code/simonwi

In [ ]:
#SCALING!
#Inspect Shape
images, labels = next(iter(train_ds))
print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Image dtype:", images.dtype)
print("\nFirst image shape:", images[0].shape)
print("First label:", labels[0])

#Next block
#check pixel values
for images, labels in train_ds.take(5):
    print(images.numpy().min(),
          images.numpy().max())

#Next block
#Scale down pixel values
train_ds_scaled = train_ds.map(lambda images, labels: (images / 255.0, labels))
val_ds_scaled = val_ds.map(lambda images, labels: (images / 255.0, labels))
test_ds_scaled = test_ds.map(lambda images, labels: (images / 255.0, labels))
external_val_ds_scaled = external_val_ds.map(lambda images, labels: (images / 255.0, labels))

#Next block
#Check that scaling has been done (shouls be between 0 and 1)
images, labels = next(iter(train_ds_scaled))
print(images.numpy().min())
print(images.numpy().max())
print(images.dtype)

Images shape: (32, 224, 224, 3)
Labels shape: (32, 4)
Image dtype: <dtype: 'float32'>

First image shape: (224, 224, 3)
First label: tf.Tensor([0. 0. 0. 1.], shape=(4,), dtype=float32)
0.0 255.0
0.0 255.0
0.0 255.0
0.0 255.0
0.0 255.0
0.0
1.0
<dtype: 'float32'>


In [ ]:
#convert every image to one-channel grayscale (instead of RGB because we already have gray images)
# this decreases the data size 3X, makes the analysis faster

train_ds_scaled = train_ds_scaled.map(lambda x, y: (tf.image.rgb_to_grayscale(x), y))
val_ds_scaled = val_ds_scaled.map(lambda x, y: (tf.image.rgb_to_grayscale(x), y))
test_ds_scaled = test_ds_scaled.map(lambda x, y: (tf.image.rgb_to_grayscale(x), y))
external_val_ds_scaled = external_val_ds_scaled.map(lambda x, y: (tf.image.rgb_to_grayscale(x), y))

In [11]:
images, labels = next(iter(train_ds_scaled))
print("Shape after grayscale conversion:", images.shape)  # expect (32, 224, 224, 1)

Shape after grayscale conversion: (32, 224, 224, 1)


In [ ]:
# this was part of the already available preprocessing pipeline
# it increasing efficieny

AUTOTUNE = tf.data.AUTOTUNE

train_ds_scaled = train_ds_scaled.prefetch(AUTOTUNE)
val_ds_scaled = val_ds_scaled.prefetch(AUTOTUNE)
test_ds_scaled = test_ds_scaled.prefetch(AUTOTUNE)
external_val_ds_scaled = external_val_ds_scaled.prefetch(AUTOTUNE)

<tf.Tensor: shape=(224, 224, 1), dtype=float32, numpy=
array([[[0.1176353],
        [0.1176353],
        [0.1176353],
        ...,
        [0.1176353],
        [0.1176353],
        [0.1176353]],

       [[0.1176353],
        [0.1176353],
        [0.1176353],
        ...,
        [0.1176353],
        [0.1176353],
        [0.1176353]],

       [[0.1176353],
        [0.1176353],
        [0.1176353],
        ...,
        [0.1176353],
        [0.1176353],
        [0.1176353]],

       ...,

       [[0.1176353],
        [0.1176353],
        [0.1176353],
        ...,
        [0.1176353],
        [0.1176353],
        [0.1176353]],

       [[0.1176353],
        [0.1176353],
        [0.1176353],
        ...,
        [0.1176353],
        [0.1176353],
        [0.1176353]],

       [[0.1176353],
        [0.1176353],
        [0.1176353],
        ...,
        [0.1176353],
        [0.1176353],
        [0.1176353]]], shape=(224, 224, 1), dtype=float32)>